# Exploring the raw data

For this demo, we've generated some data and stored it in a Unity Catalog Volume. Volumes can store any type of file and can either be managed by Unity Catalog or connected to cloud storage. Spark Declarative Pipelines can automatically pick up new files and incrementally process data in a volume making your pipelines fast and efficient.

Let's start by taking a look at the contents of the `raw_data` volume.

**Note: this notebook is a simple Exploration Notebook, it's not part of our final Pipeline!**

Having a notebook on the side to test SQL queries interactively can be very handy to accelerate exploration and build your pipelines faster!

<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-engineering&org_id=2162748966026566&notebook=%2F2-sdp-python%2Fexplorations%2F01-Exploring-the-Data&demo_name=pipeline-bike&event=VIEW&path=%2F_dbdemos%2Fdata-engineering%2Fpipeline-bike%2F2-sdp-python%2Fexplorations%2F01-Exploring-the-Data&version=1">

In [0]:
import os

raw_data_volume = "/Volumes/main/dbdemos_pipeline_bike/raw_data/"

# Print out a list of directories in our raw_data volume and a few files from those directories
for table in os.listdir(raw_data_volume):
  print(table + "/")
  for file in os.listdir(raw_data_volume + table)[:3]:
    print("  " + file)
  print("  ...")


customers_cdc/
  customers_cdc_2025-07-20.parquet
  customers_cdc_2025-07-21.parquet
  customers_cdc_2025-07-22.parquet
  ...
maintenance_logs/
  maintenance_logs_2025-07-21.csv
  maintenance_logs_2025-07-22.csv
  maintenance_logs_2025-07-23.csv
  ...
rides/
  rides_2025-07-21.csv
  rides_2025-07-22.csv
  rides_2025-07-23.csv
  ...
weather/
  weather_2025-07-21.json
  weather_2025-07-22.json
  weather_2025-07-23.json
  ...


It looks like we've got a few directories here with `csv` and `json` files in them. Let's start by taking a look at the maintenance logs files using the SQL `read_files` function.

`read_files` supports several different file formats including `csv` and `json`. Take a look at the [Databricks documentation](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files) to see the available formats and options.

Additionally, using the `STREAM` keyword `read_files` can be used in streaming tables to ingest files into Delta Lake. `read_files` leverages Auto Loader when used in a streaming table query.



In [0]:
%sql
select * from read_files("/Volumes/main/dbdemos_pipeline_bike/raw_data/maintenance_logs/*.csv", format => "csv") limit 10

maintenance_id,bike_id,reported_time,resolved_time,issue_description,_rescued_data
06a8dcaa-7c81-46de-b8b6-fbde8548719a,a5325c22-56b3-42b9-b81f-f45a92f39ec3,2025-10-20T16:50:42Z,2025-10-21,"The handlebars on my rental bike are loose and keep shifting while I'm riding, it's pretty annoying. I've tried to tighten them but it doesn't seem to be making a difference. I'd appreciate it if someone could take a look and fix the issue so I can finish my ride safely.",null
86322dee-e816-4025-a1d6-df9e951a603a,c5e47de4-27d4-460a-9754-f03cde42b020,2025-10-20T12:00:43Z,2025-10-25,"The tires on my rental bike are low on air and one of them has a slow leak, making it difficult to ride. I've only had the bike for an hour and I'm already having to stop frequently to add air. I'd appreciate it if someone could take a look at it and fix the issue as soon as possible.",null
b0d7eea8-db25-4312-b81c-c11401e7844c,e3e18217-f399-4d06-a291-9e381aa96bed,2025-10-20T23:33:47Z,2025-10-25,"The tires on my rental bike are low on air and one of them has a slow leak, making it difficult to ride. I've only had the bike for an hour and I'm already having to stop frequently to add air. I'd appreciate it if someone could take a look at it and fix the issue as soon as possible.",null
2fb4644e-2da5-4057-b1e6-73bddf5ec336,92027631-080e-44b0-b2f3-fbed89005bec,2025-10-20T15:47:51Z,2025-10-25,"The seat on my rental bike is loose and keeps sliding back and forth, making it hard to ride. I've tried to tighten it, but it doesn't seem to be staying in place. Can I get a replacement or have this one fixed ASAP?",null
e25434e8-43ae-41c5-a1f8-bfa7f16322f5,7290a980-de62-4334-a5cf-a2bacf5d7b93,2025-10-20T05:22:33Z,2025-10-22,"The seat on my rental bike is loose and keeps sliding back and forth, making it hard to ride. I've tried to tighten it, but it doesn't seem to be staying in place. Can I get a replacement or have this one fixed ASAP?",null
241fa62f-3b68-48fa-9c43-33ecbc36bb61,d8aa1100-94cf-469e-9bac-15430a040e60,2025-10-20T18:47:19Z,2025-10-23,"The handlebars on my rental bike are loose and keep shifting while I'm riding, it's pretty annoying. I've tried to tighten them but it doesn't seem to be making a difference. I'd appreciate it if someone could take a look and fix the issue so I can finish my ride safely.",null
e469784b-cb18-41a0-a80b-625a875afb34,059f91e8-1345-474f-98b3-1dc8167df796,2025-10-20T16:09:11Z,2025-10-25,"The tires on my rental bike are low on air and one of them has a slow leak, making it difficult to ride. I've only had the bike for an hour and I'm already having to stop frequently to add air. I'd appreciate it if someone could take a look at it and fix the issue as soon as possible.",null
b3101e7f-820d-4f06-b1b9-ce8078bc0ce3,712910e1-bbe5-4d10-b9d5-18a87c776378,2025-10-20T08:41:21Z,2025-10-23,"The seat on my rental bike is loose and keeps sliding back and forth, making it hard to ride. I've tried to tighten it, but it doesn't seem to be staying in place. Can I get a replacement or have this one fixed ASAP?",null
25ec3b81-2bcb-43d2-92d7-2e60aad2ab94,554e015b-c038-4fc7-afca-da2f75d9ad5b,2025-10-20T01:00:13Z,2025-10-23,"The safety reflectors on my rental bike are loose and not securely attached, which is a safety concern. I noticed the issue as soon as I started riding and I'm worried it could cause problems, especially at night. I'd appreciate it if someone could take a look and fix the reflectors as soon as possible.",null
3eede736-fbdd-4b31-af56-2cf65971e961,40a4e131-98d7-4ca5-a96b-32585f47abc0,2025-10-20T04:27:04Z,2025-10-21,"The safety reflectors on my rental bike are loose and not securely attached, which is a safety concern. I noticed the issue as soon as I started riding and I'm worried it could cause problems, especially at night. I'd appreciate it if someone could take a look and fix the reflectors as soon as possible.",null


These files contains the field `issue_description` which is a free text field people can use to enter in a description of the issue they ran into while using a bike. Free text fields often include character sequences that may break CSV parsers. Let's do some data exploration on this data to see if we are processing it correctly.

Based on our knowledge of the system giving us this data, all the fields are required. Let's look at records where that's not the case.


Yup, it looks like there's some instances where the `issue_description` fields include a newline character. We can use `multiline => true` to tell `read_files` that records may span multiple lines and see if that fixes the issue. 

In [0]:
%sql
select * from read_files("/Volumes/main/dbdemos_pipeline_bike/raw_data/maintenance_logs/*.csv", format => "csv", multiline => true)
where maintenance_id is null or bike_id is null or reported_time is null or resolved_time is null

maintenance_id,bike_id,reported_time,resolved_time,issue_description,_rescued_data



Let's do some quick spot checks on the `ride_logs` and `weather` files. The files in the `weather` directory are `json` files, so we need to make sure to use the `json` format option in `read_files`

In [0]:
%sql
select * from read_files("/Volumes/main/dbdemos_pipeline_bike/raw_data/rides/*.csv", format => "csv") limit 10

ride_id,start_time,end_time,start_station_id,end_station_id,bike_id,user_type,customer_id,_rescued_data
aa55efff-2b66-4a46-b426-4e3c0bf38ca4,2025-10-27T01:08:30Z,2025-10-27T06:34:20Z,8fd85e58-8b78-443e-8de3-9cf97871ea3e,2b954afe-d3fe-4cba-985e-7d782e110f51,1402691c-859c-4031-854c-c60e23c79d3b,member,b3ccd259-0ed2-4087-9a08-33b3d68d9029,null
76ec291c-9b55-4774-bc5b-65328d608bc3,2025-10-27T00:27:42Z,2025-10-27T03:02:18Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,d976af4c-fb83-47bd-b15c-8bbf82d7c8a7,9970bb79-20d3-409b-8a04-48d6e2098634,member,f20cadc1-1579-4d51-9b90-41f10ca8ce17,null
56d5e5f8-3aff-4900-a5d9-84a306dcfd0d,2025-10-27T04:15:03Z,2025-10-27T05:07:57Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,690a16c3-788f-456a-bab4-0628d8c08fc6,9970bb79-20d3-409b-8a04-48d6e2098634,member,7e3b44d0-c624-4d04-811c-715c80647252,null
c59540ce-a873-41a3-859b-e2e41ee360a9,2025-10-27T07:29:48Z,2025-10-27T07:30:34Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,2454bee2-5704-4831-a30a-6d20e2394899,9970bb79-20d3-409b-8a04-48d6e2098634,non-member,4532010a-b1e9-43bf-9b44-fe5cd77ec297,null
004a3377-3040-4503-ac50-0e7d70301f40,2025-10-27T08:58:23Z,2025-10-27T09:29:19Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,37ac3233-e90f-4048-af0a-ef4517371308,9970bb79-20d3-409b-8a04-48d6e2098634,member,7ffd83bc-1317-488d-b55a-076b920f6be5,null
97643944-bbfb-438e-8849-dcee8d4b322a,2025-10-27T10:03:32Z,2025-10-27T10:34:22Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,a510b3e4-61d0-4964-9d1a-23dfc3ec49b3,9970bb79-20d3-409b-8a04-48d6e2098634,member,63677e30-a762-4758-bf3a-5908ab5ce954,null
4152d413-5de3-49f5-9101-91bee991e60c,2025-10-27T12:32:37Z,2025-10-27T13:41:09Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,eee3c27b-14c8-45a8-bdbb-0053347449e8,9970bb79-20d3-409b-8a04-48d6e2098634,member,26b8c595-28c7-4441-86a4-464422aa1dde,null
35691f2b-aec6-448e-b269-9c002c05aa2d,2025-10-27T13:52:54Z,2025-10-27T16:26:07Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,ccfbdb8b-56d4-4d24-b1a4-648542053052,9970bb79-20d3-409b-8a04-48d6e2098634,member,64642f12-16a3-4421-9005-44a23ead376d,null
78c208fc-0fbc-4d7a-a8af-2e62e17fcbfd,2025-10-27T17:40:09Z,2025-10-27T18:16:46Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,fc2811e0-a882-41f4-8432-665e4c211254,9970bb79-20d3-409b-8a04-48d6e2098634,non-member,cf3c9fe0-3257-42a9-9df3-5ced9e02d5fb,null
e3917b09-8cbf-468f-be15-3e34ab64371d,2025-10-27T18:34:56Z,2025-10-27T18:38:39Z,d5c8724e-1438-4470-a5a6-9de5fec70f8e,d5c8724e-1438-4470-a5a6-9de5fec70f8e,9970bb79-20d3-409b-8a04-48d6e2098634,non-member,9c6a86e0-bbc5-4127-bfd4-712f454b957a,null


In [0]:
%sql
select * from read_files("/Volumes/main/dbdemos_pipeline_bike/raw_data/weather/*.json", format => "json") limit 10

rainfall_in,temperature_f,timestamp,wind_speed_mph,_rescued_data
0.0500000007,74.6999969482,1753056000000,22.1000003815,null
0.0099999998,74.5999984741,1753401600000,16.7000007629,null
0.0399999991,71.3000030518,1753574400000,16.2000007629,null
0.0399999991,74.4000015259,1753660800000,13.1999998093,null
0.0500000007,73.8000030518,1755216000000,18.2999992371,null
0.1299999952,76.1999969482,1755388800000,17.3999996185,null
0.0500000007,82.5999984741,1757548800000,11.3999996185,null
0.0500000007,76.6999969482,1758412800000,17.6000003815,null
0.0700000003,75.1999969482,1758672000000,10.8999996185,null
0.0599999987,71.8000030518,1759017600000,18.1000003815,null


In [0]:
%sql
select * from read_files("/Volumes/main/dbdemos_pipeline_bike/raw_data/customers_cdc/*.parquet", format => "parquet") limit 10

customer_id,user_type,registration_date,email,phone,age_group,membership_tier,preferred_payment,home_station_id,is_active,operation,event_timestamp,_rescued_data
26777f90-ca16-4c65-8ace-5789ac939cb0,member,12-28-2024 20:51:20,member2@bikerent.com,555-7235,18-25,basic,credit_card,fc2811e0-a882-41f4-8432-665e4c211254,true,APPEND,09-04-2025 20:51:20,null
13c06f62-207f-492e-9e51-3493ca587372,member,02-26-2025 20:51:20,member87@bikerent.com,555-1029,55+,enterprise,cash,ceb50c29-1ca9-4cbb-b17a-0766f7fb81a9,true,APPEND,09-04-2025 20:51:20,null
7f205d40-6e91-4d59-9a9e-e1c99bcccdbc,member,02-21-2025 20:51:20,member88@bikerent.com,555-1583,55+,basic,mobile_pay,37ac3233-e90f-4048-af0a-ef4517371308,true,APPEND,09-04-2025 20:51:20,null
26b8c595-28c7-4441-86a4-464422aa1dde,member,10-31-2023 20:51:20,member100@bikerent.com,555-6562,18-25,premium,credit_card,5a83249e-f528-4051-9430-78cdfd5aa0b7,true,APPEND,09-04-2025 20:51:20,null
ecb833e8-2e29-4429-a2e5-b7bd9475b6a4,member,12-30-2023 20:51:20,member203@bikerent.com,555-9554,46-55,basic,credit_card,fc2811e0-a882-41f4-8432-665e4c211254,true,APPEND,09-04-2025 20:51:20,null
9860b834-9c31-4e6b-92c7-dfd7781a5d1d,member,03-11-2024 20:51:20,member219@bikerent.com,555-6492,36-45,enterprise,mobile_pay,68f2d56f-d3d3-4c18-abaf-0a6a5bb867bc,true,APPEND,09-04-2025 20:51:20,null
010c4af5-a1ae-4fd8-8a2c-a14f05dc9010,member,12-31-2023 20:51:20,member304@bikerent.com,555-4434,26-35,enterprise,mobile_pay,68f2d56f-d3d3-4c18-abaf-0a6a5bb867bc,true,APPEND,09-04-2025 20:51:20,null
5664cb19-947d-4234-99c1-32440c4f7146,member,12-21-2024 20:51:20,member382@bikerent.com,555-2036,18-25,premium,credit_card,eee3c27b-14c8-45a8-bdbb-0053347449e8,true,APPEND,09-04-2025 20:51:20,null
7c398d7d-e877-47f5-a209-c3a7a769f712,member,09-05-2024 20:51:20,member433@bikerent.com,555-4692,46-55,enterprise,cash,ccfbdb8b-56d4-4d24-b1a4-648542053052,true,APPEND,09-04-2025 20:51:20,null
2ecf2154-142d-4c2f-ab1e-12449aaec003,member,08-17-2024 20:51:20,member485@bikerent.com,555-5468,18-25,enterprise,credit_card,690a16c3-788f-456a-bab4-0628d8c08fc6,true,APPEND,09-04-2025 20:51:20,null
